Célula 1 - Carregamento com Alinhamento Dinâmico de Colunas

In [1]:
import pandas as pd
import numpy as np
import glob
import os

# 1. Carregar arquivos do DNIT mapeando o padrão de arquivos
caminho_dnit = '../data/raw/dnit/*pa*.csv'
arquivos_dnit = glob.glob(caminho_dnit)

lista_dfs_dnit = []
for arq in arquivos_dnit:
    # Alterado para encoding='latin-1' para ler corretamente os acentos da base original
    df_ano = pd.read_csv(arq, sep=';', encoding='latin-1', low_memory=False)
    
    # Padroniza as colunas antes de juntar para garantir que o Pandas alinhe os nomes iguais
    df_ano.columns = df_ano.columns.str.lower().str.replace(' ', '_')
    lista_dfs_dnit.append(df_ano)

# 2. Concatenação completa: Mantém TODAS as colunas de todas as tabelas
df_dnit = pd.concat(lista_dfs_dnit, ignore_index=True)

print("Base unificada do DNIT carregada com sucesso!")
print(f"Formato total com todas as colunas preservadas: {df_dnit.shape}")

Base unificada do DNIT carregada com sucesso!
Formato total com todas as colunas preservadas: (46390, 18)


Célula 2 - Filtro Fino e Imputação da Série Temporal

In [ ]:
# 1. Lista exata de colunas solicitadas (já em minúsculas conforme padrão da Célula 1)
colunas_desejadas = [
    'id_malha', 'uf', 'contrato', 'ano', 'mes', 'rodovia', 'km', 'sentido',
    'km_inicial', 'km_final', 'num_faixas', 'superfície', 'data_aval.',
    'ip', 'ic', 'icm', 'icm_unificado'
]

# Filtra mantendo apenas as colunas que existem na base
colunas_presentes = [col for col in colunas_desejadas if col in df_dnit.columns]
df_dnit_filtrado = df_dnit[colunas_presentes].copy()

# 2. Garantir tipos numéricos para chaves de cruzamento e ordenação
df_dnit_filtrado['km'] = pd.to_numeric(df_dnit_filtrado['km'].astype(str).str.replace(',', '.'), errors='coerce')
df_dnit_filtrado = df_dnit_filtrado.dropna(subset=['rodovia', 'km'])

df_dnit_filtrado['ano'] = pd.to_numeric(df_dnit_filtrado['ano'], errors='coerce')
df_dnit_filtrado['mes'] = pd.to_numeric(df_dnit_filtrado['mes'], errors='coerce')

# Ordenação cronológica e espacial indispensável
df_dnit_filtrado = df_dnit_filtrado.sort_values(by=['rodovia', 'km', 'ano', 'mes'])

# 3. Imputação Categórica (Textos)
# Das colunas que você escolheu, superfície e sentido são os textos principais
colunas_textos = ['superfície', 'sentido']
for col in colunas_textos:
    if col in df_dnit_filtrado.columns:
        df_dnit_filtrado[col] = df_dnit_filtrado.groupby(['rodovia', 'km'])[col].ffill()
        df_dnit_filtrado[col] = df_dnit_filtrado.groupby(['rodovia', 'km'])[col].bfill()
        df_dnit_filtrado[col] = df_dnit_filtrado[col].fillna('Não Mapeado')

# 4. Imputação Numérica (Converte vírgula para ponto e interpola)
colunas_indices = ['ic', 'ip', 'icm', 'icm_unificado', 'num_faixas']
for col in colunas_indices:
    if col in df_dnit_filtrado.columns:
        df_dnit_filtrado[col] = df_dnit_filtrado[col].astype(str).str.replace(',', '.')
        df_dnit_filtrado[col] = pd.to_numeric(df_dnit_filtrado[col], errors='coerce')
        
        # Interpolação baseada no KM
        df_dnit_filtrado[col] = df_dnit_filtrado.groupby(['rodovia', 'km'])[col].transform(lambda x: x.interpolate(method='linear'))
        df_dnit_filtrado[col] = df_dnit_filtrado.groupby(['rodovia', 'km'])[col].bfill().ffill()

# 5. Remover duplicatas de medições repetidas para consolidar o trecho
df_dnit_final = df_dnit_filtrado.drop_duplicates(subset=['rodovia', 'km'])

# 6. Exportação do Checkpoint (latin-1)
import os
os.makedirs('../data/processed', exist_ok=True)
df_dnit_final.to_csv('../data/processed/dnit_limpo.csv', index=False, sep=';', encoding='latin-1')

print(f"Base do DNIT estritamente filtrada e salva! Total de colunas: {len(df_dnit_final.columns)}")
print("Colunas mantidas:", df_dnit_final.columns.tolist())

Base do DNIT processada salvando a totalidade das colunas!
Formato final exportado em latin-1: (3913, 18)
